In [3]:
!pip install datasets
!pip install transformers
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which

In [ ]:
import json
import os
from datasets import load_dataset, Dataset
from peft import get_peft_model, LoraConfig, TaskType
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling
)
import torch
import random

In [ ]:
import os
import json
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.model_selection import train_test_split

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "saved_models/m2_model"
MAX_LENGTH = 512
INSTRUCTION = "Analyze the following argument and identify any logical fallacies:"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token


def load_base_dataset():
    return load_dataset("MidhunKanadan/logical-fallacy-classification", split="train")


def load_none_samples():
    facts = [
        "The Earth revolves around the Sun.",
        "Water boils at 100 degrees Celsius at sea level.",
        "Photosynthesis occurs in the chloroplasts of plant cells.",
        "Gravity pulls objects toward the Earth’s center.",
        "The Pacific Ocean is the largest ocean on Earth.",
        "Sound travels slower than light.",
        "A triangle has three sides.",
        "Dogs are domesticated animals.",
        "Electricity is conducted by metals like copper.",
        "Bananas are a good source of potassium.",
        "Reading regularly can expand your vocabulary.",
        "Vaccines help protect against diseases.",
        "The Moon affects the tides of Earth."
    ]
    return [{"statement": fact, "label": "None"} for fact in facts]


def augment_dataset(dataset, new_samples):
    for sample in new_samples:
        dataset = dataset.add_item(sample)
    return dataset


def build_label_mappings(labels):
    unique_labels = sorted(set(labels))
    label2id = {label: idx for idx, label in enumerate(unique_labels)}
    id2label = {idx: label for label, idx in label2id.items()}
    return label2id, id2label


def save_label_mappings(label2id, id2label):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(f"{OUTPUT_DIR}/label2id.json", "w") as f:
        json.dump(label2id, f)
    with open(f"{OUTPUT_DIR}/id2label.json", "w") as f:
        json.dump(id2label, f)


def format_example(example, label2id):
    input_text = example["statement"]
    label_id = label2id[example["label"]]
    return {
        "text": f"<|user|>\n{INSTRUCTION}\n\n{input_text}\n<|assistant|>\nFallacy ID: {label_id}",
        "label": label_id
    }


def tokenize_sample(sample):
    return tokenizer(sample["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)


def main():
    dataset = load_base_dataset()
    none_samples = load_none_samples()
    dataset = augment_dataset(dataset, none_samples)

    labels = dataset["label"]
    label2id, id2label = build_label_mappings(labels)
    save_label_mappings(label2id, id2label)

    formatted_dataset = dataset.map(lambda x: format_example(x, label2id))

    tokenized_dataset = formatted_dataset.map(tokenize_sample)

    split = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = split["train"]
    eval_dataset = split["test"]

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        load_in_4bit=True,
        device_map="auto"
    )

    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, lora_config)

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        logging_dir=f"{OUTPUT_DIR}/logs",
        logging_steps=10,
        num_train_epochs=3,
        warmup_steps=50,
        weight_decay=0.01,
        save_total_limit=2,
        fp16=True,
        report_to="none"
    )

    data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator
    )

    trainer.train()


    trainer.save_model("./tinyllama-fallacy/final")
    tokenizer.save_pretrained("./tinyllama-fallacy/final")

    !zip -r tokenizer_files.zip /content/tinyllama-fallacy/final

    from google.colab import files
    files.download("tokenizer_files.zip")
    print("Training complete. Model saved!")


if __name__ == "__main__":
    main()


tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.98k [00:00<?, ?B/s]

...

/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/

  adding: content/tinyllama-fallacy/final/ (stored 0%)
  adding: content/tinyllama-fallacy/final/tokenizer_config.json (deflated 68%)
  adding: content/tinyllama-fallacy/final/special_tokens_map.json (deflated 73%)
  adding: content/tinyllama-fallacy/final/adapter_config.json (deflated 53%)
  adding: content/tinyllama-fallacy/final/tokenizer.model (deflated 55%)
  adding: content/tinyllama-fallacy/final/training_args.bin (deflated 51%)
  adding: content/tinyllama-fallacy/final/README.md (deflated 66%)
  adding: content/tinyllama-fallacy/final/adapter_model.safetensors (deflated 8%)
  adding: content/tinyllama-fallacy/final/tokenizer.json (deflated 85%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Training complete. Model saved!


In [ ]:
def generate_prompt():
    prompt = """You are a logic expert. For each statement, choose exactly one logical fallacy from the list below that best matches the reasoning error. If there is no fallacy, reply with "none".

Choose from:
- Appeal to nature
- Appeal to worse problems
- False dilemma
- Hasty generalization
- Slippery slope
- Appeal to authority
- Appeal to majority
- Appeal to tradition
- none

Examples:
"""
    return prompt

In [2]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system ==

In [5]:
import torch
from transformers import pipeline
from datasets import load_dataset
from tqdm import tqdm
import re
from sklearn.metrics import classification_report

pipe = pipeline("text-generation", model="./tinyllama-fallacy-finetuned", torch_dtype=torch.bfloat16, device_map="auto")

def generate_prompt():
    prompt = """You are a logic expert. For each statement, identify exactly one logical fallacy from the list below that best matches the reasoning error. If there is no fallacy, reply with "none".


FALLACIES:
- Appeal to nature: Judgment is based solely on whether the subject of judgment is “natural” or “unnatural.”
- Appeal to worse problems: Dismissing an argument or complaint due to what are perceived to be more important problems
- False dilemma: Presenting only two options when more exist
- Hasty generalization: Drawing broad conclusions from a small sample
- Slippery slope: Claiming one action will inevitably lead to extreme negative consequences
- Appeal to authority: Using an expert of dubious credentials or using only one opinion to promote a product or idea.
- Appeal to majority: Claiming something is true/right because many people believe it
- Appeal to tradition: Arguing something is good/right because it's traditional or has been done for a long time

EXAMPLES:
"""
    return prompt

dataset = load_dataset("mithrandir22/cocolofa", split="validation")
dataset = dataset.rename_column("comment", "input")
dataset = dataset.rename_column("fallacy", "label")
texts = dataset["input"]
labels = dataset["label"]
data = [
    ("Senator Randall isn't lying when she says she cares about her constituents—she is a senator so she wouldn't lie to people she cares about.", "Appeal to authority"),
    ("I love eating burgers.", "none"),
    ("If we ban Hummers because they are bad for the environment, eventually the government will ban all cars, so we should not ban Hummers.", "False Dilemma"),
]

def build_training_prompt(data):
    prompt = generate_prompt()
    for text, fallacy in data:
        prompt += f"Text: {text}\nFallacy: {fallacy}\n\n"
    return prompt

base_prompt = build_training_prompt(data)
output = []

for test in tqdm(texts, desc="Classifying fallacies"):
    full_prompt = f"{base_prompt}Text: {test}\nFallacy:"
    print(full_prompt)
    outputs = pipe(full_prompt, max_new_tokens=5, do_sample=True, temperature=0.1, top_k=20, top_p=0.8)
    llm_output = outputs[0]["generated_text"]
    pairs = {}
    blocks = re.split(r'\n(?=Text:)', llm_output)
    for block in blocks:
        text_match = re.search(r'Text:\s*(.+)', block)
        fallacy_match = re.search(r'Fallacy:\s*(.+)', block)
        if text_match and fallacy_match:
            text = text_match.group(1).strip()
            fallacy = fallacy_match.group(1).strip()
            pairs[text] = fallacy
    output.append(pairs.get(test, "Fallacy not found"))

output = [label.lower() for label in output]
print(classification_report(labels, output))

Classifying fallacies:   0%|          | 0/1538 [00:00<?, ?it/s]

You are a logic expert. For each statement, identify exactly one logical fallacy from the list below that best matches the reasoning error. If there is no fallacy, reply with "none".


FALLACIES:
- Appeal to nature: Judgment is based solely on whether the subject of judgment is “natural” or “unnatural.”
- Appeal to worse problems: Dismissing an argument or complaint due to what are perceived to be more important problems
- False dilemma: Presenting only two options when more exist
- Hasty generalization: Drawing broad conclusions from a small sample
- Slippery slope: Claiming one action will inevitably lead to extreme negative consequences
- Appeal to authority: Using an expert of dubious credentials or using only one opinion to promote a product or idea.
- Appeal to majority: Claiming something is true/right because many people believe it
- Appeal to tradition: Arguing something is good/right because it's traditional or has been done for a long time

EXAMPLES:
Text: Senator Randall is

NameError: name 'pipe' is not defined

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

dataset = load_dataset("mithrandir22/cocolofa", split="train")
dataset = dataset.rename_column("comment", "input")
dataset = dataset.rename_column("fallacy", "label")

def generate_prompt(example):
    return (
        "You are a logic expert. For each statement, identify exactly one logical fallacy from the list below that best matches the reasoning error. "
        'If there is no fallacy, reply with "none".\n\n'
        "FALLACIES:\n"
        "- Appeal to nature: Judgment is based solely on whether the subject of judgment is “natural” or “unnatural.\n"
        "- Appeal to worse problems: Dismissing an argument or complaint due to what are perceived to be more important problems\n"
        "- False dilemma: Presenting only two options when more exist\n"
        "- Hasty generalization: Drawing broad conclusions from a small sample\n"
        "- Slippery slope: Claiming one action will inevitably lead to extreme negative consequences\n"
        "- Appeal to authority: Using an expert of dubious credentials or using only one opinion to promote a product or idea.\n"
        "- Appeal to majority: Claiming something is true/right because many people believe it\n"
        "- Appeal to tradition: Arguing something is good/right because it's traditional or has been done for a long time\n\n"
        f"Text: {example['input']}\nFallacy: {example['label']}"
    )

dataset = dataset.map(lambda x: {"text": generate_prompt(x)})

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./tinyllama-fallacy-finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    bf16=True,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

trainer.save_model("./tinyllama-fallacy-finetuned")
tokenizer.save_pretrained("./tinyllama-fallacy-finetuned")


Map:   0%|          | 0/5370 [00:00<?, ? examples/s]

Map:   0%|          | 0/5370 [00:00<?, ? examples/s]

<ipython-input-4-3eb5e57463f8>:68: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
50,1.224800
100,0.654600
150,0.644700
200,0.646500
250,0.636700
300,0.632800
350,0.611500
400,0.487700
450,0.479700
500,0.490000


('./tinyllama-fallacy-finetuned/tokenizer_config.json',
 './tinyllama-fallacy-finetuned/special_tokens_map.json',
 './tinyllama-fallacy-finetuned/tokenizer.model',
 './tinyllama-fallacy-finetuned/added_tokens.json',
 './tinyllama-fallacy-finetuned/tokenizer.json')

In [8]:
!zip -r tokenizer_files.zip /content/tinyllama-fallacy/final

from google.colab import files
files.download("tokenizer_files.zip")
print("Training complete. Model saved!")

	zip warning: name not matched: /content/tinyllama-fallacy/final

zip error: Nothing to do! (try: zip -r tokenizer_files.zip . -i /content/tinyllama-fallacy/final)


FileNotFoundError: Cannot find file: tokenizer_files.zip

In [10]:
from huggingface_hub import HfApi
from google.colab import userdata
# userdata.get('HF_TOKEN')

api = HfApi(token=userdata.get('HF_TOKEN'))
api.upload_folder(
    folder_path="./tinyllama-fallacy-finetuned",
    repo_id="mithrandir22/tinyLLama-Logical-Fallacy",
    repo_type="model",
)

optimizer.pt:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

rng_state.pth:   0%|          | 0.00/14.2k [00:00<?, ?B/s]

Upload 15 LFS files:   0%|          | 0/15 [00:00<?, ?it/s]

scheduler.pt:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

...

training_args.bin:   0%|          | 0.00/5.24k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.24k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/mithrandir22/tinyLLama-Logical-Fallacy/commit/5babec2c30e0e3ac3054c4798344fce8d3fbafc2', commit_message='Upload folder using huggingface_hub', commit_description='', oid='5babec2c30e0e3ac3054c4798344fce8d3fbafc2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/mithrandir22/tinyLLama-Logical-Fallacy', endpoint='https://huggingface.co', repo_type='model', repo_id='mithrandir22/tinyLLama-Logical-Fallacy'), pr_revision=None, pr_num=None)

In [5]:
!pip install datasets

In [2]:
from datasets import load_dataset
dataset = load_dataset("mithrandir22/cocolofa", split="test")
dataset = dataset.rename_column("comment", "input")
dataset = dataset.rename_column("fallacy", "label")
texts = dataset["input"]
labels = dataset["label"]
data = [
    ("Senator Randall isn't lying when she says she cares about her constituents—she wouldn't lie to people she cares about.", "Circular Reasoning"),
    ("I love eating burgers.", "none"),
    ("If we ban Hummers because they are bad for the environment, eventually the government will ban all cars, so we should not ban Hummers.", "False Dilemma"),
]

In [6]:
from tqdm import tqdm
import re
def build_training_prompt(data):
    prompt = generate_prompt()
    for text, fallacy in data:
        prompt += f"Text: {text}\nFallacy: {fallacy}\n\n"
    return prompt

base_prompt = build_training_prompt(data)
output = []

for test in tqdm(texts, desc="Classifying fallacies"):
    full_prompt = f"{base_prompt}Text: {test}\nFallacy:"
    outputs = pipe(full_prompt, max_new_tokens=5, do_sample=True, temperature=0.1, top_k=20, top_p=0.8)
    llm_output = outputs[0]["generated_text"]
    pairs = {}
    blocks = re.split(r'\n(?=Text:)', llm_output)
    for block in blocks:
        text_match = re.search(r'Text:\s*(.+)', block)
        fallacy_match = re.search(r'Fallacy:\s*(.+)', block)
        if text_match and fallacy_match:
            text = text_match.group(1).strip()
            fallacy = fallacy_match.group(1).strip()
            pairs[text] = fallacy
    output.append(pairs.get(test, "Fallacy not found"))

Classifying fallacies: 100%|██████████| 798/798 [02:02<00:00,  6.54it/s]


In [9]:
from sklearn.metrics import classification_report
output = [label.lower() for label in output]
print(classification_report(labels, output))

                          precision    recall  f1-score   support

        appeal to author       0.00      0.00      0.00         0
     appeal to authority       0.00      0.00      0.00        56
        appeal to better       0.00      0.00      0.00         0
      appeal to majority       1.00      0.02      0.03        59
        appeal to nature       0.08      0.87      0.15        55
           appeal to the       0.00      0.00      0.00         0
          appeal to trad       0.00      0.00      0.00         0
     appeal to tradition       0.00      0.00      0.00        59
           appeal to wor       0.00      0.00      0.00         0
         appeal to worse       0.00      0.00      0.00         0
appeal to worse problems       0.00      0.00      0.00        64
       fallacy not found       0.00      0.00      0.00         0
           false dilemma       0.00      0.00      0.00        50
       false equivalence       0.00      0.00      0.00         0
    hasty

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_

In [ ]:
import os
import json
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.model_selection import train_test_split

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "saved_models/m2_model"
MAX_LENGTH = 512
INSTRUCTION = "Analyze the following argument and identify any logical fallacies:"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token


def load_base_dataset():
    return load_dataset("mithrandir22/cocolofa", split="train")


def build_label_mappings(labels):
    unique_labels = sorted(set(labels))
    label2id = {label: idx for idx, label in enumerate(unique_labels)}
    id2label = {idx: label for label, idx in label2id.items()}
    return label2id, id2label


def save_label_mappings(label2id, id2label):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(f"{OUTPUT_DIR}/label2id.json", "w") as f:
        json.dump(label2id, f)
    with open(f"{OUTPUT_DIR}/id2label.json", "w") as f:
        json.dump(id2label, f)


def format_example(example, label2id):
    input_text = example["statement"]
    label_id = label2id[example["label"]]
    return {
        "text": f"<|user|>\n{INSTRUCTION}\n\n{input_text}\n<|assistant|>\nFallacy ID: {label_id}",
        "label": label_id
    }


def tokenize_sample(sample):
    return tokenizer(sample["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)


def main():
    dataset = load_base_dataset()
    none_samples = load_none_samples()
    dataset = augment_dataset(dataset, none_samples)

    labels = dataset["label"]
    label2id, id2label = build_label_mappings(labels)
    save_label_mappings(label2id, id2label)

    formatted_dataset = dataset.map(lambda x: format_example(x, label2id))

    tokenized_dataset = formatted_dataset.map(tokenize_sample)

    split = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = split["train"]
    eval_dataset = split["test"]

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        load_in_4bit=True,
        device_map="auto"
    )

    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, lora_config)

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        logging_dir=f"{OUTPUT_DIR}/logs",
        logging_steps=100,
        num_train_epochs=3,
        warmup_steps=50,
        weight_decay=0.01,
        save_total_limit=2,
        fp16=True,
        report_to="none"
    )

    data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator
    )

    trainer.train()


    trainer.save_model("./tinyllama-fallacy/final")
    tokenizer.save_pretrained("./tinyllama-fallacy/final")

    !zip -r tokenizer_files.zip /content/tinyllama-fallacy/final

    from google.colab import files
    files.download("tokenizer_files.zip")
    print("Training complete. Model saved!")


if __name__ == "__main__":
    main()


Step,Training Loss
10,3.192900
20,3.110000
30,2.659300
40,2.717000
50,2.602700
60,2.699600
70,2.484700
80,2.690300
90,2.567000
100,2.622600


('./tinyllama-fallacy/final/tokenizer_config.json',
 './tinyllama-fallacy/final/special_tokens_map.json',
 './tinyllama-fallacy/final/tokenizer.model',
 './tinyllama-fallacy/final/added_tokens.json',
 './tinyllama-fallacy/final/tokenizer.json')